# OriginalWoodelf — Depth Sweep

Runs **OriginalWoodelf** (`woodelf.simple_woodelf` — cube-based AAAI algorithm)
across all datasets and task types in the `woodelfhd_depth_sweep_experiment`.  
This notebook is **independent** and can run in parallel with the other method notebooks.


### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` repo
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method original_woodelf`
5. Writes partial results to Drive as `original_woodelf.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | Google Drive parquet |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

> **Runtime estimate:** OriginalWoodelf crashes at high depths (D≥18 for SHAP, D≥15
> for interactions), so most deep-depth entries are MEMORY_CRASH (instant).  
> Expect a few hours on a standard Colab CPU runtime.

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Must match the DRIVE_FOLDER used in notebooks 01, 03, and 04.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'original_woodelf.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

Results will be saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/original_woodelf.json


In [ ]:
# ── Step 3: Clone repository ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'  # e.g. https://github.com/ron-wettenstein/WoodelfExperiments.git

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (352/352), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 352 (delta 222), reused 262 (delta 135), pack-reused 0 (from 0)
Receiving objects: 100% (352/352), 210.16 KiB | 1.88 MiB/s, done.
Resolving deltas: 100% (222/222), done.


In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install woodelf_explainer

!pip install -q -e /content/treebranchmarks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:00
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
!pip uninstall -y treebranchmarks

Found existing installation: treebranchmarks 0.1.0
Uninstalling treebranchmarks-0.1.0:
  Successfully uninstalled treebranchmarks-0.1.0


In [ ]:
!pip install -q -e /content/treebranchmarks

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
# The framework writes the method cache file to Drive after every approach result.
# On restart, copy it back to the local cache directory so the experiment
# skips already-completed entries and only runs what is still missing.
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'original_woodelf.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

Restored method cache (5 KB)


In [ ]:
# ── Step 6: Run the experiment (OriginalWoodelf only) ─────────────────────────
# --method original_woodelf : only the OriginalWoodelfApproach is timed
# The method name 'original_woodelf' matches ORIGINAL_WOODELF.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method original_woodelf \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']

  > D=6  n=118108  m=0
  [approach:OriginalWoodelf] CACHED=6.611s

  > D=9  n=118108  m=0
  [approach:OriginalWoodelf] CACHED=69.878s

  > D=12  n=118108  m=0
  [approach:OriginalWoodelf] CACHED=4282.393s

  > D=15  n=118108  m=0
  [approach:OriginalWoodelf] CACHED=212073.720s

  > D=18  n=118108  m=0
  [approach:OriginalWoodelf] MEMORY CRASH (configured)

  > D=21  n=118108  m=0
  [approach:OriginalWoodelf] MEMORY CRASH (configured)

Mission: fraud_detection BG SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=472432
  tasks     : ['Background SHAP']

  > D=6  n=118108  m=472432
  [approach:OriginalWoodelf] CACHED=13.738s

  > D=9  n=118108  m=472432
  [approach:OriginalWoodelf] CACHED=83.462s

  > D=12  n=118108  m=472432
  [approach:OriginalWoodelf] CACH

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')

Entries in method cache: 41
Sample entry: Path-Dependent SHAP n=118108 m=0 D=6 T=100  →  6.611s

File saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/original_woodelf.json


# Old Runs

In [ ]:
# ── Step 6: Run the experiment (OriginalWoodelf only) ─────────────────────────
# --method original_woodelf : only the OriginalWoodelfApproach is timed
# The method name 'original_woodelf' matches ORIGINAL_WOODELF.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method original_woodelf \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 167MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 14.84s — T=100, D=6, L=32.0, F=397
  [approach:OriginalWoodelf] CACHED=6.611s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 17.84s — T=100, D=9, L=82.8, F=397
  [approach:OriginalWoodelf] CACHED=69.878s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 8.48s — T=10, D=12, L=202.5, F=397
  [approach:OriginalWoodelf] CACHED=4282.393s

 